## prepare


In [1]:
import numpy as np
if not hasattr(np, 'float_'):
    np.float_ = np.float64


import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch.utils.data import TensorDataset
import datasets
from datasets import load_from_disk, Image
import torch
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from torchmetrics.image.fid import FrechetInceptionDistance
import math
import copy

In [2]:
import os
import random
import numpy as np
import torch


def set_seed(seed: int):
    # 1. Python random
    random.seed(seed)

    # 2. NumPy
    np.random.seed(seed)

    # 3. PyTorch (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果你用多个 GPU

    # 4. cuDNN 设置为确定性模式，关闭 benchmark
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True





def seed_worker(worker_id):
    # 每个 worker 基于相同或偏移后的种子
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


# 在脚本最上方调用一次
SEED = 42
set_seed(SEED)

# PyTorch 1.7+ 可直接用 generator
gen = torch.Generator()
gen.manual_seed(SEED)

### dataset

In [3]:
cuda_num = 1
s = 28
p = 1
INPUT_DIM = p * s * s
LATENT_DIM = 5
HIDDEN_DIM = 500
data_folder = "data_mnist"
datasetname = "mnist"

# train+label
train_dataset = datasets.load_from_disk(f"/home/yzm/{data_folder}/train")
print(f"Total samples: {len(train_dataset)}")
train_images = (
    torch.stack([torch.tensor(np.array(img)) for img in train_dataset["image"]]).float()
    / 255.0
)
train_images = train_images.reshape(-1, 28 * 28) # Flatten to [60000, 784]
train_labels = torch.tensor(train_dataset["label"], dtype=torch.long)
train_dataset_torch = TensorDataset(train_images, train_labels)



# test+label
test_dataset = datasets.load_from_disk(f"/home/yzm/{data_folder}/test")
test_images = (
    torch.stack([torch.tensor(np.array(img)) for img in test_dataset["image"]]).float()
    / 255.0
)
test_images = test_images.reshape(-1, 28 * 28) # Flatten to [10000, 784]
test_labels = torch.tensor(test_dataset["label"], dtype=torch.long)
test_dataset_torch = TensorDataset(test_images, test_labels)


Total samples: 60000


In [4]:
from torch.utils.data import random_split

# 定义划分比例
train_size = int(0.9 * len(train_dataset_torch))  # 90%
val_size = len(train_dataset_torch) - train_size  # 10%

# 按比例划分数据集
train_dataset_torch, val_dataset_torch = random_split(train_dataset_torch, [train_size, val_size], generator=gen)

# 打印划分后的数据集大小
print(f"Train dataset size: {len(train_dataset_torch)}")
print(f"Validation dataset size: {len(val_dataset_torch)}")

Train dataset size: 54000
Validation dataset size: 6000


### vae class


In [5]:
class VAE_Encoder(nn.Module):
    """
    VAE的encoder，将输入的图片映射到隐变量的分布，输出隐变量分布的均值和方差.
    由于方差不能为负，所以这里输出的是log(方差)，而不是方差本身.
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE_Encoder, self).__init__()

        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.Tanh())

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)


class VAE_Decoder(nn.Module):
    """
    VAE的decoder，将隐变量映射回图片.
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE_Decoder, self).__init__()

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.decoder(z)


class VAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM):
        super(VAE, self).__init__()

        self.latent_dim = latent_dim

        self.encoder = VAE_Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = VAE_Decoder(input_dim, hidden_dim, latent_dim)

    def encode(self, x):
        """
        将输入的图片映射到隐变量的分布，输出隐变量分布的均值和log(方差).
        """
        return self.encoder(x)

    def decode(self, z):
        """
        将隐变量映射回图片.
        """
        return self.decoder(z)

    def reparameterize(self, mu, logvar):
        """
        重参数化技巧，先从标准正态分布中采样一个epsilon，然后根据隐变量分布的均值和方差，计算出隐变量.
        """
        std = torch.exp(
            0.5 * logvar
        )  # 计算标准差, std = sqrt(var) = sqrt(exp(logvar)) = exp(logvar/2)
        epsilon = torch.randn_like(
            std, requires_grad=False
        )  # 从标准正态分布中采样epsilon
        return mu + epsilon * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

### loss


In [6]:
import torch
import math

def loss_iwae_gauss(mu, logvar, x, decoder, k=5, sigma2=1.0, device=None):
    if device is None:
        device = x.device

    B = x.size(0)
    D = x.view(B, -1).size(1)
    z_dim = mu.size(-1)

    # 采样 k 个 z: shape [B, k, Z]
    eps = torch.randn(B, k, z_dim, device=device)
    std = torch.exp(0.5 * logvar)                          # [B, Z]
    z = mu.unsqueeze(1) + std.unsqueeze(1) * eps           # [B, k, Z]

    # 常量
    dtype = z.dtype
    log_2pi = torch.tensor(math.log(2 * math.pi), device=device, dtype=dtype)
    log_k = torch.tensor(math.log(k), device=device, dtype=dtype)

    # 第二项 log p(z) 标准正态先验: [B, k]
    log_pz = -0.5 * (z**2 + log_2pi).sum(-1)

    # 第三项 log q(z|x): [B, k]
    log_qz = -0.5 * (
        ((z - mu.unsqueeze(1)) ** 2) / (std.unsqueeze(1) ** 2)
        + logvar.unsqueeze(1)
        + log_2pi
    ).sum(-1)

    # 解码: flatten z -> [B*k, Z] and decode -> [B*k, D]
    z_flat = z.view(B * k, z_dim)
    x_recon = decoder(z_flat)                                # [B*k, D]
    x_expand = x.unsqueeze(1).expand(-1, k, -1).contiguous().view(B * k, -1)
    diff = x_expand - x_recon                                # [B*k, D]

    # 第一项 log p(x|z) 似然，固定方差 sigma2: [B, k]
    const_px = torch.tensor(0.5 * D * math.log(2 * math.pi * sigma2), device=device, dtype=diff.dtype)
    log_px = (-0.5 * (diff**2 / sigma2).sum(-1) - const_px).view(B, k)

    # importance weights: log-sum-exp 聚合 （per data point: [B]）
    log_w = log_px + log_pz - log_qz                        # [B, k]
    m = log_w.max(dim=1, keepdim=True).values               # [B, 1]
    sum_exp = torch.exp(log_w - m).sum(dim=1)               # [B]
    Lk = m.squeeze(1) + torch.log(sum_exp) - log_k          # [B]

    total_loss = - Lk.sum()
    return total_loss


In [7]:
def loss_function(x, recon_x, mu, logvar):
    # 这里要计算ELBO(也就是论文中的$\mathcal{L}$)，但是由于论文中的目标是最大化ELBO，pytorch中是最小化loss，所以这里实际计算的是-ELBO
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())  # 计算KL散度
    BCE = nn.functional.binary_cross_entropy(
        recon_x, x, reduction="sum"
    )  # 计算重构误差，对应论文中的$-\log p_{\theta}(x|z)$，注意BCE loss本身前面有个负号
    MSE = nn.functional.mse_loss(recon_x, x, reduction="sum")
    return KLD


In [8]:
def score_fn_torch(z: torch.Tensor, x_target: torch.Tensor, decoder: nn.Module, sigma2=0.1) -> torch.Tensor:
    """
    计算 ∇_z log p(z, x_target)，完全基于 PyTorch。

    参数：
        z: [B, D]，需要设置 requires_grad=True
        x_target: [784] 或 [1, 784]，float tensor
        decoder: PyTorch 解码器
        sigma2: 高斯观测模型的方差
    返回：
        score: [B, D]，对应每个 z 的梯度
    """
    assert z.requires_grad, "z must have requires_grad=True"

    B,D = z.shape[0], z.shape[1]
    if x_target.ndim == 1:
        x_target = x_target.unsqueeze(0)
    
    x_target = x_target.view(B, -1)   # [B, 3072]
    x_rec = decoder(z).view(B, -1)
    diff = x_target - x_rec

    # log p(z) = -0.5 * ||z||^2
    logpz = -0.5 * (z ** 2).sum(dim=1)

    # log p(x|z) = -1/(2σ²) * ||x - decoder(z)||²
    logpx = -0.5 / sigma2 * (diff ** 2).sum(dim=1)

    # —— 分别求两项梯度 ——
    # 注意：sum() 后才对 z 求梯度
    grad_pz = -z  # [B, D]
    grad_px = torch.autograd.grad(logpx.sum(), z, create_graph=True, retain_graph=True)[
        0
    ]  # [B, D]

    # 合并两项得到 g(z) = ∇ log p(z) + ∇ log p(x|z)
    g = grad_pz + grad_px  # [B, D]

    return g

In [9]:
def bamloss_batch(
    mu, logvar, x_target, decoder,
    sample_K=2, lambda_t=0.1, sigma2=0.1,
    writer=None, global_step=None,
):
    """
    向量化的 BaM 损失，支持 batch 并行。
    参数：
        mu: [N, D] torch tensor
        logvar: [N, D] torch tensor
        x_target: [N, *] torch tensor
        decoder: decoder 网络（用于 score_fn 计算）
        sample_K: 每个样本采样 K 个 z
        lambda_t: 正则化系数
        sigma2: 生成模型噪声方差
    返回：
        标量 loss（对 batch 求平均）
    """
    N, D = mu.shape

    # 1. 重参数化采样 z: [N, K, D]
    std = torch.exp(0.5 * logvar)
    var = std**2
    eps = torch.randn(N, sample_K, D, device=mu.device)
    z = mu.unsqueeze(1) + eps * std.unsqueeze(1)
    z.requires_grad_(True)

    # 2. 计算 score，先 reshape 到 [N*K, D]
    z_flat = z.view(N * sample_K, D)
    x_rep = x_target.unsqueeze(1)               \
                     .expand(N, sample_K, *x_target.shape[1:]) \
                     .reshape(N * sample_K, *x_target.shape[1:])
    scores_flat = score_fn_torch(z_flat, x_rep, decoder, sigma2)
    scores = scores_flat.view(N, sample_K, D)

    # z_bar = z.mean(dim=1)        # [N, D]
    # g_bar = scores.mean(dim=1)   # [N, D]

    # 3. 重构误差（每个 z）
    # x_rec = decoder(z_flat).view(N, sample_K, -1)
    # recon_error = ((x_rep.view(N, sample_K, -1) - x_rec) ** 2).sum(dim=2)  # [N, K]

    # 4. 逐 z_l 计算
    # g_norm2 = (scores**2).sum(dim=2)  # [N, K]
    g_norm2_sigma = (var.unsqueeze(1) * scores**2).sum(dim=2)  # [N, K]
    align_term = ((z - mu.unsqueeze(1)) * scores).sum(dim=2)  # [N, K]

    # 5. Score-matching 三项（逐样本）V3

    # 1) tr(I) = D
    term1 = D
    # 2) 1/L ∑_l g_l^T Σ g_l = 1/L ∑_l sum_d var_d * g_{l,d}^2
    term2 = g_norm2_sigma.mean(dim=1)  # [N]
    # 3) 2/L ∑_l (z_l - mu)^T g_l
    term3 = 2 * align_term.mean(dim=1)  # [N]

    #score_div = term1 + term2 + term3  # [N]
    score_div =  term2 + term3

    # 6. 最终 loss 对 batch 求平均
    loss = score_div.mean()  # 对 batch 求平均

    return loss

In [10]:
def cv_bamloss_batch(
    mu, logvar, x_target, decoder,
    sample_K1=2,      # 原始估计量的样本数
    sample_K2=10,     # CV 控制变量的样本数
    lambda_t=0.1, sigma2=0.1,
    writer=None, global_step=None,
):
    """
    向量化的 BaM 损失 + Control Variates (修复版)。
    
    参数：
        sample_K1: 原始项的样本数 (用于最终估计)
        sample_K2: CV 项的样本数 (用于估计最优系数 c)
    """
    N, D = mu.shape
    eps = 1e-5

    # -------------------------------------------------------------------------
    # [Control Variates 准备阶段] 计算 g(mu) 和 grad(f1(mu))
    # -------------------------------------------------------------------------
    # 确保 mu 计算梯度
    mu.requires_grad_(True)
    
    # 1. 计算 g(mu) [N, D]
    # 注意：这里直接使用 x_target，因为 mu 对应每个 batch 只有 1 个
    g_mu = score_fn_torch(mu, x_target, decoder, sigma2)

    # 2. 计算 f1(mu) = g(mu)^T Sigma g(mu)
    std = torch.exp(0.5 * logvar)
    var = std**2
    f1_mu = (var * g_mu**2).sum(dim=1) # [N]

    # 3. 计算 grad(f1(mu)) [N, D]
    grad_f1_mu = torch.autograd.grad(
        outputs=f1_mu.sum(), inputs=mu, create_graph=True
    )[0]

    # -------------------------------------------------------------------------
    # [原始项采样] 使用 K1 个样本
    # -------------------------------------------------------------------------
    # 重参数化采样 z: [N, K1, D]
    eps_z1 = torch.randn(N, sample_K1, D, device=mu.device)
    z1 = mu.unsqueeze(1) + eps_z1 * std.unsqueeze(1)  # [N, K1, D]
    z1.requires_grad_(True)

    # 计算 score
    z1_flat = z1.view(N * sample_K1, D)
    x_rep1 = x_target.unsqueeze(1) \
                     .expand(N, sample_K1, *x_target.shape[1:]) \
                     .reshape(N * sample_K1, *x_target.shape[1:])
    scores1_flat = score_fn_torch(z1_flat, x_rep1, decoder, sigma2)
    scores1 = scores1_flat.view(N, sample_K1, D)

    z1_minus_mu = z1 - mu.unsqueeze(1)  # [N, K1, D]

    # 原始项
    g_norm2_sigma_raw = (var.unsqueeze(1) * scores1**2).sum(dim=2)  # [N, K1]
    align_term_raw = (z1_minus_mu * scores1).sum(dim=2)              # [N, K1]
    
    # -------------------------------------------------------------------------
    # [CV 项采样] 使用 K2 个独立样本
    # -------------------------------------------------------------------------
    # 控制变量项 (CV Terms) [N, K2, D]
    eps_z2 = torch.randn(N, sample_K2, D, device=mu.device)
    z2 = mu.unsqueeze(1) + eps_z2 * std.unsqueeze(1)  # [N, K2, D]
    z2.requires_grad_(True)

    # 计算 score
    z2_flat = z2.view(N * sample_K2, D)
    x_rep2 = x_target.unsqueeze(1) \
                     .expand(N, sample_K2, *x_target.shape[1:]) \
                     .reshape(N * sample_K2, *x_target.shape[1:])
    scores2_flat = score_fn_torch(z2_flat, x_rep2, decoder, sigma2)
    scores2 = scores2_flat.view(N, sample_K2, D)

    z2_minus_mu = z2 - mu.unsqueeze(1)  # [N, K2, D]

    # CV 控制变量值
    # CV1: grad(f1(mu))^T (z - mu)
    # CV2: g(mu)^T (z - mu)
    cv1_vals = (grad_f1_mu.unsqueeze(1) * z2_minus_mu).sum(dim=2)  # [N, K2]
    cv2_vals = (g_mu.unsqueeze(1) * z2_minus_mu).sum(dim=2)        # [N, K2]

    # CV 对应的原始项值 (用于估计相关性)
    g_norm2_sigma_cv = (var.unsqueeze(1) * scores2**2).sum(dim=2)  # [N, K2]
    align_term_cv = (z2_minus_mu * scores2).sum(dim=2)              # [N, K2]

    # -------------------------------------------------------------------------
    # [估计最优系数 c]
    # -------------------------------------------------------------------------

    # 计算最优系数 c1, c2 (对 K 维度求统计量) [N]
    # c = Cov(Term, CV) / Var(CV)
    def compute_optimal_c(term_vals, cv_vals):
        # 中心化
        term_centered = term_vals - term_vals.mean(dim=1, keepdim=True)
        cv_centered = cv_vals - cv_vals.mean(dim=1, keepdim=True)
        
        # Covariance / Variance
        numerator = (term_centered * cv_centered).mean(dim=1)
        denominator = cv_vals.var(dim=1, unbiased=False) + eps
        
        c = numerator / denominator
        # c 只作为降低方差的系数，不希望优化器去通过改变 c 来优化 Loss
        c = c.detach()  # 不参与梯度优化
        # 如果 c 计算出来是几千几万，说明 CV 和原始项的相关性出了问题或者方差估计完全错误。
        # 限制在 [-100, 100] 或更小的范围内，保证训练初期 Loss 不崩。
        # c = torch.clamp(c, min=-100.0, max=100.0)  # 防止数值爆炸
        
        return c

        
    c1 = compute_optimal_c(g_norm2_sigma_cv, cv1_vals)  # [N]
    c2 = compute_optimal_c(align_term_cv, cv2_vals)      # [N]

    # -------------------------------------------------------------------------
    # [应用 CV 校正到 K1 样本]
    # -------------------------------------------------------------------------
    # 注意：CV 项用 K2 估计的 c，但应用到 K1 的样本上
    # 需要重新计算 K1 样本对应的 CV 值
    cv1_vals_k1 = (grad_f1_mu.unsqueeze(1) * z1_minus_mu).sum(dim=2)  # [N, K1]
    cv2_vals_k1 = (g_mu.unsqueeze(1) * z1_minus_mu).sum(dim=2)        # [N, K1]

    # -------------------------------------------------------------------------
    # [Score-matching 三项 (应用 CV)]
    # -------------------------------------------------------------------------
    
    # 1) tr(I) = D
    term1 = D
    
    # 2) Term 2: Mean(Raw - c1 * CV)
    # 1/L ∑ ||g_l||_Σ^2 - c1 * 1/L ∑ CV1
    term2_corrected = g_norm2_sigma_raw - c1.unsqueeze(1) * cv1_vals_k1
    term2 = term2_corrected.mean(dim=1)  # [N]
    
    
    # 3) Term 3: Mean(Raw - c2 * CV) * 2
    # 2/L ∑ (z_l - mu)^T g_l - 2 * c2 * 1/L ∑ CV2
    term3_corrected = align_term_raw - c2.unsqueeze(1) * cv2_vals_k1
    term3 = 2 * term3_corrected.mean(dim=1)  # [N]


    # 汇总 Loss
    score_div = term2 + term3

    # 6. 最终 loss 对 batch 求平均
    loss = score_div.mean() # 对 batch 求平均
    
    return loss

### eval

In [11]:
def eval_fid(model, test_loader, device, fid_metric):
    model.eval()
    fid_metric.reset()

    with torch.no_grad():
        for real, _ in test_loader:
            real = real.to(device)

            # 编码—采样—解码
            mu, logvar = model.encode(real)
            z = model.reparameterize(mu, logvar)
            fake = model.decode(z)

            # 从 [B, D] 恢复成 [B,3,side,side]
            B, D = real.shape
            side = int(D**0.5)
            real_img = real.view(B, 1, side, side).repeat(1, 3, 1, 1)
            fake_img = fake.view(B, 1, side, side).repeat(1, 3, 1, 1)

            # resize 到 299×299
            real_rs = F.interpolate(
                real_img, size=(299, 299), mode="bilinear", align_corners=False
            )
            fake_rs = F.interpolate(
                fake_img, size=(299, 299), mode="bilinear", align_corners=False
            )

            # 更新 FID 统计（uint8）
            fid_metric.update((real_rs * 255).to(torch.uint8), real=True)
            fid_metric.update((fake_rs * 255).to(torch.uint8), real=False)

    # 计算并返回得分
    fid_score = fid_metric.compute().item()
    model.train()
    return fid_score

In [12]:
# def eval_fid(model, test_loader, device, fid_metric):
#     model.eval()
#     fid_metric.reset()

#     with torch.no_grad():
#         for real, _ in test_loader:
#             real = real.to(device)

#             # 编码—采样—解码
#             mu, logvar = model.encode(real)
#             z = model.reparameterize(mu, logvar)
#             fake = model.decode(z)

#             # 从 [B, D] 恢复成 [B,3,side,side]
#             B, D = real.shape
#             #side = int(D**0.5)
#             real_img = real.view(B, 3, s, s)   # 如果 flatten 过，需要 reshape 回来
#             fake_img = fake.view(B, 3, s, s)


#             # resize 到 299×299
#             real_rs = nn.functional.interpolate(real_img, size=(299, 299), mode="bilinear", align_corners=False)
#             fake_rs = nn.functional.interpolate(fake_img, size=(299, 299), mode="bilinear", align_corners=False)


#             # 更新 FID 统计（uint8）
#             fid_metric.update((real_rs * 255).to(torch.uint8), real=True)
#             fid_metric.update((fake_rs * 255).to(torch.uint8), real=False)

#     # 计算并返回得分
#     fid_score = fid_metric.compute().item()
#     model.train()
#     return fid_score

In [13]:
import math
@torch.no_grad()
def evaluate_iwae_gauss(model, dataloader, device, k=5000, sigma2=1.0):
    total_loglik = 0.0
    total_samples = 0
    D = None

    for x, _ in dataloader:
        x = x.to(device)
        B, D = x.shape
        total_samples += B

        # 编码
        mu, logvar = model.encode(x)  # [B, Z]
        std = torch.exp(0.5 * logvar)  # [B, Z]

        # 采样 k 个 z
        eps = torch.randn(B, k, mu.size(1), device=device)
        z = mu.unsqueeze(1) + std.unsqueeze(1) * eps  # [B, k, Z]

        # 第二项：log p(z) 标准正态先验
        log_pz = -0.5 * (z**2 + math.log(2 * math.pi)).sum(-1)  # [B, k]

        # 第三项：log q(z|x)
        log_qz = -0.5 * (
            ((z - mu.unsqueeze(1)) ** 2) / (std.unsqueeze(1) ** 2)
            + logvar.unsqueeze(1)
            + math.log(2 * math.pi)
        ).sum(
            -1
        )  # [B, k]

        # 解码
        x_recon = model.decode(z.view(B * k, -1))  # [B*k, D]
        x_expand = x.unsqueeze(1).repeat(1, k, 1).view(B * k, -1)
        diff = x_expand - x_recon  # [B*k, D]

        # 第一项：log p(x|z) 似然，固定方差 sigma2
        log_px = (
            -0.5 * (diff**2 / sigma2).sum(-1) - 0.5 * D * math.log(2 * math.pi * sigma2)
        ).view(
            B, k
        )  # [B, k]

        # importance weights: log-sum-exp 聚合
        log_w = log_px + log_pz - log_qz  # [B, k]
        m = log_w.max(dim=1, keepdim=True).values  # [B, 1]
        Lk = (
            m.squeeze(1) + torch.log(torch.exp(log_w - m).sum(dim=1)) - math.log(k)
        )  # [B]

        total_loglik += Lk.sum().item()

    avg_Lk = total_loglik / total_samples
    avg_NLL = -avg_Lk
    bpd = avg_NLL / (D * math.log(2))  # bits/dim
    return avg_Lk, avg_NLL, bpd

### plt

In [14]:
# gary
def write_test_images(writer, model, data_loader, device, global_step, sample_K=7,folder_name="img"):
    """
    将测试输入和重构结果保存到 TensorBoard。

    参数：
    - writer: SummaryWriter 实例
    - model: VAE 模型
    - data_loader: DataLoader，用于获取一个 batch
    - device: 运行设备（cuda or cpu）
    - global_step: TensorBoard 全局步数
    - sample_K: 最多展示的样本数
    """
    model.eval()
    batch, _ = next(iter(data_loader))
    batch = batch.to(device)
    K = min(sample_K, batch.size(0))
    inputs = batch[:K].cpu().view(K, 28, 28)

    # 编码 -> 采样 -> 解码
    with torch.no_grad():
        mu, logvar = model.encode(batch)
        sigma = torch.exp(0.5 * logvar)
        eps = torch.randn_like(sigma)

        z = mu + eps * sigma
        recon = model.decode(z[:K]).cpu().view(K, 28, 28).numpy()

    # 合并原图和重构图到一张图
    fig = plt.figure(figsize=(15, 6))
    for i in range(K):
        # 第一行：原图
        ax = fig.add_subplot(2, K, i + 1)
        ax.imshow(inputs[i], cmap="gray")
        ax.axis("off")
        # 第二行：重构图
        ax = fig.add_subplot(2, K, K + i + 1)
        ax.imshow(recon[i], cmap="gray")
        ax.axis("off")

    writer.add_figure(f"{folder_name}/{global_step}-rec", fig, global_step)
    plt.close(fig)

In [15]:
# # RGB
# def write_test_images(
#     writer,
#     model,
#     data_loader,
#     device,
#     global_step,
#     sample_K=7,
#     img_size=(s, s),
#     num_channels=3,
#     folder_name="img",
# ):
#     """
#     在 TensorBoard 上展示 CIFAR-10 原图 & VAE 重构图。

#     参数
#     - writer: SummaryWriter
#     - model: VAE（输入维度 3072）
#     - data_loader: DataLoader，产出 (flat_img, label)，flat_img shape [B,3072]
#     - device: torch.device
#     - global_step: 整数步数
#     - sample_K: 最多展示多少张
#     - img_size: (H, W)，这里 (32,32)
#     - num_channels: 通道数，这里 3
#     """
#     model.eval()

#     # 1) 从 data_loader 累积扁平张量 flat_imgs 直到 >= sample_K
#     all_flat = []
#     for flat_img, _ in data_loader:
#         all_flat.append(flat_img)
#         if sum(x.size(0) for x in all_flat) >= sample_K:
#             break
#     flat_imgs = torch.cat(all_flat, dim=0).to(device)  # 至少 sample_K 张

#     B = flat_imgs.size(0)
#     K = min(sample_K, B)
#     C, H, W = num_channels, *img_size

#     # 2) 把前 K 张先恢复成 [K, C, H, W] 供可视化
#     inputs = flat_imgs[:K].view(K, C, H, W).cpu()

#     # 3) 编码 → 样本 → 解码（全程基于扁平向量）
#     with torch.no_grad():
#         mu, logvar = model.encode(flat_imgs)  # [B, latent_dim]×2
#         sigma = torch.exp(0.5 * logvar)
#         eps = torch.randn_like(sigma)
#         z = mu + eps * sigma

#         recon_flat = model.decode(z[:K]).cpu()  # [K, 3072]
#         recon = recon_flat.view(K, C, H, W)  # 还原成 [K, C, H, W]

#     # 4) 转成 HWC NumPy 数组
#     inputs_np = inputs.permute(0, 2, 3, 1).contiguous().numpy()  # [K, H, W, C]
#     recon_np = recon.permute(0, 2, 3, 1).contiguous().numpy()  # [K, H, W, C]

#     # 5) 按组绘图，每组最多 7 张
#     group_size = 7
#     for group_idx in range(0, K, group_size):
#         end_idx = min(group_idx + group_size, K)
#         current_K = end_idx - group_idx

#         fig = plt.figure(figsize=(2 * current_K, 4))
#         for i in range(current_K):
#             # 上排：原图
#             ax1 = fig.add_subplot(2, current_K, i + 1)
#             ax1.imshow(inputs_np[group_idx + i])
#             ax1.axis("off")

#             # 下排：重构图
#             ax2 = fig.add_subplot(2, current_K, current_K + i + 1)
#             ax2.imshow(recon_np[group_idx + i])
#             ax2.axis("off")

#         writer.add_figure(
#             f"{folder_name}/{global_step}-{group_idx // group_size}",
#             fig,
#             global_step,
#         )
#         plt.close(fig)
#     model.train()

In [16]:
def log_latent_space(writer, model, dataloader, global_step, device):
    model.eval()
    all_mu = []
    all_logvar = []
    all_labels = []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            mu, logvar = model.encode(x)
            all_mu.append(mu.cpu())
            all_logvar.append(logvar.cpu())
            all_labels.extend(y.cpu().tolist())

    all_mu = torch.cat(all_mu, dim=0)  # [N, z_dim]
    all_logvar = torch.cat(all_logvar, dim=0)  # [N, z_dim]
    all_labels = [str(l) for l in all_labels]  # 转成字符串，方便 Color by

    max_points = 2000
    all_mu = all_mu[:max_points]
    all_labels = all_labels[:max_points]
    all_logvar = all_logvar[:max_points]

    # print("mu 是否有nan: ", torch.isnan(all_mu).any(), torch.isinf(all_mu).any())
    # print("logvar 是否有nan: ", torch.isnan(all_logvar).any(), torch.isinf(all_logvar).any())
    # print("数据维度: ", all_mu.shape, all_logvar.shape, len(all_labels))
    # print("数据范围: ", all_mu.min().item(), all_mu.max().item(), all_logvar.min().item(), all_logvar.max().item())
    # print("数据方差: ", all_mu.std(dim=0), all_logvar.std(dim=0))

    # print("标签类别: ", set(all_labels))
    # 写入 TensorBoard Embedding
    writer.add_embedding(
        mat=all_mu,
        metadata=all_labels,  # 标签（比如数字类别）
        tag="latent_mu",
        global_step=global_step,
    )
    writer.add_embedding(
        mat=all_logvar,
        metadata=all_labels,  # 标签（比如数字类别）
        tag="latent_logvar",
        global_step=global_step,
    )

In [17]:
def log_latent_stats(model, loader, writer, epoch, device):
    model.eval()
    ############## 写 mu/std/z m##############
    all_mu, all_std, all_z = [], [], []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            mu, logvar = model.encode(x)  # [B, Z]
            std = torch.exp(0.5 * logvar)  # [B, Z]
            # 如果你想按重参数化采样的 z
            eps = torch.randn_like(std)
            z = mu + eps * std  # [B, Z]

            all_mu.append(mu.cpu())
            all_std.append(std.cpu())
            all_z.append(z.cpu())

    # 拼接成 [N, Z]
    all_mu = torch.cat(all_mu, dim=0)
    all_std = torch.cat(all_std, dim=0)
    all_z = torch.cat(all_z, dim=0)

    # 对样本维度取均值 → [Z]
    mu_mean = all_mu.mean(dim=0)  # Tensor of shape [Z]
    std_mean = all_std.mean(dim=0)
    z_mean = all_z.mean(dim=0)

    # 控制台打印（可选）
    print(f"Epoch {epoch+1}  mu_mean per dim:  {mu_mean.tolist()}")
    print(f"Epoch {epoch+1}  std_mean per dim: {std_mean.tolist()}")
    print(f"Epoch {epoch+1}  z_mean per dim:   {z_mean.tolist()}")

    # 写入 TensorBoard：每个潜变量维度一个 scalar
    for i in range(mu_mean.size(0)):
        writer.add_scalar(f"latent/mu_dim{i}", mu_mean[i].item(), epoch)
        writer.add_scalar(f"latent/std_dim{i}", std_mean[i].item(), epoch)
        writer.add_scalar(f"latent/z_dim{i}", z_mean[i].item(), epoch)

    # 如果还想看分布，可写 histogram：
    writer.add_histogram("histogram/mu", all_mu, epoch)
    writer.add_histogram("histogram/std", all_std, epoch)
    writer.add_histogram("histogram/z", all_z, epoch)

    ############## 写 g_dim ##############
    Z = all_mu.size(1)  # 潜变量维度 5

    for d in range(Z):
        vals = epoch_metrics_g[f"g_abs_dim{d}"]  # list of length num_batches
        avg = sum(vals) / len(vals)  # 维度 d 上的平均梯度
        writer.add_scalar(f"latent_g/g_abs_dim{d}", avg, epoch)

        vals = epoch_metrics_g[f"g_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g/g_dim{d}", avg, epoch)

    # 写 g 的整体分布直方图
    all_g = []
    for d in range(Z):
        all_g.extend(epoch_metrics_g[f"g_dim{d}"])  # 收集所有维度的 g

    all_g_tensor = torch.tensor(all_g, dtype=torch.float32)
    writer.add_histogram("histogram_g/g", all_g_tensor, epoch)

    print(f"Epoch {epoch+1}  g per dim: {[epoch_metrics_g[f'g_abs_dim{d}'][-1] for d in range(Z)]}")

    ############### 写 g 的分解项 ##############
    for d in range(Z):
        vals = epoch_metrics_g[f"g_pz_abs_dim{d}"]  # list of length num_batches
        avg = sum(vals) / len(vals)  # 维度 d 上的平均
        writer.add_scalar(f"latent_g_pz/g_pz_abs_dim{d}", avg, epoch)
        vals = epoch_metrics_g[f"g_px_abs_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_px/g_px_abs_dim{d}", avg, epoch)

        vals = epoch_metrics_g[f"g_pz_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_pz/g_pz_dim{d}", avg, epoch)
        vals = epoch_metrics_g[f"g_px_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_px/g_px_dim{d}", avg, epoch)

    # g_pz 和 g_px 的整体分布直方图
    all_g_pz = []
    all_g_px = []
    for d in range(Z):            
        all_g_pz.extend(epoch_metrics_g[f"g_pz_dim{d}"])
        all_g_px.extend(epoch_metrics_g[f"g_px_dim{d}"])

    all_g_pz_tensor = torch.tensor(all_g_pz, dtype=torch.float32)
    all_g_px_tensor = torch.tensor(all_g_px, dtype=torch.float32)
    writer.add_histogram("histogram_g/g_pz", all_g_pz_tensor, epoch)
    writer.add_histogram("histogram_g/g_px", all_g_px_tensor, epoch)

    print(f"Epoch {epoch+1}  g_pz per dim: {[epoch_metrics_g[f'g_pz_abs_dim{d}'][-1] for d in range(Z)]}")
    print(f"Epoch {epoch+1}  g_px per dim: {[epoch_metrics_g[f'g_px_abs_dim{d}'][-1] for d in range(Z)]}")

    model.train()  # 别忘了恢复训练模式

## run mnist


### ours

In [18]:
#### ours
batch_size0 = 128
batch_size_test = 64
batch_size_test_nll = 16
epoch0 = 100
lr0 = 1e-4


sigma20 = 1
lambda_t0 = 0.1
lambda_KLt = 0.1

lambda_KL0 = 0
lambda_sum = 0.5 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1}
lambda_KL0 = 0 # vae的 KL，非vae时=0

torch.cuda.empty_cache()

device = torch.device(f"cuda:{cuda_num}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

flag_vae = False
flag_iwae = False
flag_cv = False
prefix = f"rerun5-F3-anew-latent{LATENT_DIM}"
folder = "/home/yzm/ipy/runs__re"
file_idx = 141
sample_K0 = 2

lambda_D0 = 0.5
lambda_sum = 1 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1} 的分母
num_KLt = 1 # 消融改这里

for lr0 in [1e-3]:
    for sample_K0 in [10]:  
        for qqq in range(5):    
            for tensround in range(file_idx, file_idx + 1):
                time_now = datetime.now().strftime("%m-%d_%H-%M")
                if flag_vae:
                    lambda_sum = 0
                    lambda_KLt = 1
                    lambda_KL0 = 1
                    model_dir=f"{folder}/{prefix}-{datasetname}-vae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir 
                    )
                elif flag_iwae:
                    model_dir=f"{folder}/{prefix}-{datasetname}-iwae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                elif flag_cv:
                    model_dir=f"{folder}/{prefix}-{datasetname}-cv-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                else:
                    model_dir=f"{folder}/{prefix}-{datasetname}-ours-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{num_KLt/lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                    log_dir=model_dir
                )
                model = VAE().to(device)
                model_t = VAE().to(device)
                optimizer = torch.optim.Adam(model.parameters(), lr=lr0)

                fid_metric = FrechetInceptionDistance(feature=2048).to(device)

                data_loader = DataLoader(
                    train_dataset_torch, batch_size=batch_size0, shuffle=True,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                val_loader = DataLoader(
                    val_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4, worker_init_fn=seed_worker, generator=gen
                )
                                

                test_loader = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                test_loader_nll = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test_nll, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )

                plt_ck_p = epoch0 // 20  # 显示 20 次测试结果

                total_steps = epoch0 * len(data_loader)

                for epoch in range(epoch0):

                    losses = []
                    Dqs = []
                    KLs = []
                    KL_ts = []
                    MSEs = []
                    BCEs = []
                    sum_Dq_plus_lKLt_dot_KLt_s = []

                    alpha_max = 1.0
                    beta_max = 1.0
                    gamma_max = 1.0

                    warmup1 = 30      # 只用 MSE+KL
                    warmup2 = 80      # 完整过渡到 MSE+KL+BaM
                    # alpha = alpha_max * min(1.0, epoch / warmup1)
                    alpha = alpha_max
                    gamma = gamma_max
                    beta = beta_max
                    # beta = beta_max * min(
                    #     1.0, max(0.0, (epoch - warmup1) / (warmup2 - warmup1))
                    # )
                    # k 控制增长速度，k越小增长越慢

                    # k = 0.1
                    # epoch_since_warmup = max(0, epoch - warmup1)
                    # beta = beta_max * (1 - math.exp(-k * epoch_since_warmup))
                    # 确保在warmup2时beta接近beta_max

                    # 在 epoch 开头，保存 old encoder 参数
                    old_enc_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.encoder.state_dict().items()
                    }

                    for batch_idx, (x, y) in enumerate(data_loader):
                        x = x.to(device)  # [batch, ...]
                        recon_batch, mu, logvar = model(x)  # [batch, D]

                        # 恢复旧 encoder 的参数到临时 old_encoder 推断（在 CPU 拷贝上不需要梯度）
                        with torch.no_grad():
                            # 将保存的 old state 恢复到 model.encoder 的一个临时副本计算 mu_old/logvar_old
                            # 更高效：直接在当前 encoder 上 load_state_dict(old_enc_state)，但要小心线程安全与 GPU；这里用简单方式：
                            # 创建新 encoder 实例会更健壮但也更重 —— 下面采用直接把旧参数载入 model.encoder 并再恢复当前参数的办法（较快）
                            model_t.encoder.load_state_dict(old_enc_state)   # load old weights
                            # 注意：mu_old, logvar_old 都是在 no_grad 环境下计算的（无需保持 grad）
                            mu_old, logvar_old = model_t.encoder(x)         # compute old outputs

                        global_step = epoch * len(data_loader) + batch_idx

                        # KL = 1.0 / x.size(0) * loss_function(x, recon_batch, mu, logvar)
                        # KLs.append(KL.item())
                        
                        if not flag_vae and not flag_iwae:                            
                            if flag_cv:
                                Dq = cv_bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K1=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                            else:
                                Dq = bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                        else:
                            Dq = torch.tensor(0.0)
                        Dqs.append(Dq.item())
                        
                        MSE = nn.functional.mse_loss(
                            recon_batch, x, reduction="sum"
                        ) / x.size(0)

                        MSEs.append(MSE.item())
                        MSE = sigma20 * MSE / sigma20

                        var = torch.exp(logvar)
                        var_old = torch.exp(logvar_old)
                        # 先在 latent 维度 D 上求和
                        KL_t_per_latent = 0.5 * (
                            torch.log(var / var_old) +
                            (var_old + (mu_old - mu).pow(2)) / var - 1
                        ).sum(dim=1)                        # [N]
                        # 再对 batch 维度求均值
                        KL_t = KL_t_per_latent.mean()        # scalar

                        KL_ts.append(KL_t.item())

                        Dq *= lambda_D0
                        # KL *= lambda_KL0
                        KL_t *= num_KLt / lambda_KLt

                        sum_Dq_plus_lKLt_dot_KLt = Dq + KL_t
                        sum_Dq_plus_lKLt_dot_KLt_s.append(sum_Dq_plus_lKLt_dot_KLt.item())
                        sum_Dq_plus_lKLt_dot_KLt *= lambda_sum
                        
                        if flag_iwae:
                            loss = 1 / x.size(0) * loss_iwae_gauss(mu, logvar, x, model.decode, k=sample_K0)
                        else:
                            #loss = MSE + alpha * KL + beta * sum_Dq_plus_lKLt_dot_KLt
                            loss = MSE + beta * sum_Dq_plus_lKLt_dot_KLt

                        # BCE = nn.functional.binary_cross_entropy(
                        #     recon_batch, x, reduction="sum"
                        # )
                        # BCE = BCE / x.size(0)
                        # BCEs.append(BCE.item())

                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                        losses.append(loss.item())

                    # 3. 记录每个 epoch 的平均 loss
                    mean_loss = np.mean(losses)
                    mean_MSE = np.mean(MSEs)
                    # mean_BCE = np.mean(BCEs)
                    mean_Dq = np.mean(Dqs)
                    # mean_KL = np.mean(KLs)
                    mean_KL_t = np.mean(KL_ts)
                    mean_Dq_plus_lKLt_dot_KLt = np.mean(sum_Dq_plus_lKLt_dot_KLt_s)

                    writer.add_scalar("Loss/train_epoch", mean_loss, epoch)
                    writer.add_scalar("MSE/train_epoch", mean_MSE, epoch)
                    # writer.add_scalar("BCE/train_epoch", mean_BCE, epoch)
                    writer.add_scalar("Dq/train_epoch", mean_Dq, epoch)
                    # writer.add_scalar("KL/train_epoch", mean_KL, epoch)
                    writer.add_scalar("KL_t/train_epoch", mean_KL_t, epoch)
                    writer.add_scalar("Sum/train_epoch", mean_Dq_plus_lKLt_dot_KLt, epoch)

                    write_test_images(writer, model, test_loader, device, epoch)
                    
                    fid_score = eval_fid(model, val_loader, device, fid_metric)
                    writer.add_scalar("FID/val_epoch", fid_score, epoch)
                    print(
                        f"[Epoch {epoch+1}] Loss: {mean_loss:.4f}, FID: {fid_score:.4f}, BCE: {'N/A'}"
                    )
                    print(
                        f"           MSE: {mean_MSE:.4f}, Dq: {mean_Dq:.4f}, KL: {'N/A'}, KL_t: {mean_KL_t:.4f}"
                    )
                    torch.cuda.empty_cache()
                    
                write_test_images(
                    writer,
                    model,
                    test_loader,
                    device,
                    global_step=0,
                    sample_K=42,
                    folder_name="img_final_epoch",
                )
                
                fid_score_final = eval_fid(model, test_loader, device, fid_metric)
                writer.add_scalar("FID/test_epoch", fid_score_final, epoch)
                print(f"Final Test FID: {fid_score_final:.4f}")

                hparam_dict = {
                    "batch_size": batch_size0,
                    "epochs": epoch0,
                    "lr": lr0,
                    "K": sample_K0,
                    "sigma2": sigma20,
                    "lambda_D": lambda_D0,
                    "lambda_KL": lambda_KL0,
                    "lambda_KLt": lambda_KLt,
                    "lambda_sum": lambda_sum,
                    "tensround": tensround,
                }
                metric_dict = {
                    "last loss": mean_loss,
                    "last FID": float(fid_score_final),
                    "last MSE": mean_MSE,
                }
                writer.add_hparams(hparam_dict, metric_dict)
                # 4. 训练结束后关闭 writer
                writer.close()
                
                torch.save(
                    model,
                    f"{model_dir}/final_model.pt",
                )
                
                # Lk, NLL, bpd = evaluate_iwae_gauss(model, test_loader_nll, device)
                # writer.add_scalar("NLL/test_final_epoch", NLL, 99)
                # writer.add_scalar("bpd/test_final_epoch", bpd, 99)


Using device: cuda:1


/home/yzm/anaconda3/envs/gen/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Metric `FrechetInceptionDistance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


[Epoch 1] Loss: 38.2625, FID: 147.1991, BCE: N/A
           MSE: 38.1260, Dq: -1.9212, KL: N/A, KL_t: 54.8589
[Epoch 2] Loss: 27.3941, FID: 104.1980, BCE: N/A
           MSE: 27.6208, Dq: -0.8195, KL: N/A, KL_t: 9.1570
[Epoch 3] Loss: 24.7425, FID: 90.2937, BCE: N/A
           MSE: 24.9718, Dq: -0.6356, KL: N/A, KL_t: 4.4251
[Epoch 4] Loss: 23.2744, FID: 80.3769, BCE: N/A
           MSE: 23.4959, Dq: -0.5780, KL: N/A, KL_t: 3.3753
[Epoch 5] Loss: 22.3376, FID: 74.7186, BCE: N/A
           MSE: 22.5390, Dq: -0.5269, KL: N/A, KL_t: 3.1028
[Epoch 6] Loss: 21.6488, FID: 70.8335, BCE: N/A
           MSE: 21.8341, Dq: -0.4876, KL: N/A, KL_t: 2.9256
[Epoch 7] Loss: 21.1388, FID: 66.6285, BCE: N/A
           MSE: 21.3143, Dq: -0.4693, KL: N/A, KL_t: 2.9604
[Epoch 8] Loss: 20.7227, FID: 63.3475, BCE: N/A
           MSE: 20.8911, Dq: -0.4607, KL: N/A, KL_t: 3.0970
[Epoch 9] Loss: 20.3651, FID: 61.8901, BCE: N/A
           MSE: 20.5315, Dq: -0.4507, KL: N/A, KL_t: 2.9443
[Epoch 10] Loss: 20.0574,

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-bjsnu155'


[Epoch 49] Loss: 16.7503, FID: 40.3819, BCE: N/A
           MSE: 16.8692, Dq: -0.3778, KL: N/A, KL_t: 3.4984
[Epoch 50] Loss: 16.7288, FID: 40.0922, BCE: N/A
           MSE: 16.8440, Dq: -0.3754, KL: N/A, KL_t: 3.6218
[Epoch 51] Loss: 16.6795, FID: 39.7256, BCE: N/A
           MSE: 16.8011, Dq: -0.3820, KL: N/A, KL_t: 3.4683
[Epoch 52] Loss: 16.6675, FID: 39.8110, BCE: N/A
           MSE: 16.7970, Dq: -0.4010, KL: N/A, KL_t: 3.5491
[Epoch 53] Loss: 16.6471, FID: 39.9279, BCE: N/A
           MSE: 16.7757, Dq: -0.4003, KL: N/A, KL_t: 3.5741
[Epoch 54] Loss: 16.6308, FID: 39.2033, BCE: N/A
           MSE: 16.7525, Dq: -0.3974, KL: N/A, KL_t: 3.8496


...

[Epoch 23] Loss: 17.9764, FID: 47.7507, BCE: N/A
           MSE: 18.0910, Dq: -0.3660, KL: N/A, KL_t: 3.4177
[Epoch 24] Loss: 17.8931, FID: 46.4709, BCE: N/A
           MSE: 18.0028, Dq: -0.3604, KL: N/A, KL_t: 3.5241
[Epoch 25] Loss: 17.8005, FID: 45.0023, BCE: N/A
           MSE: 17.9038, Dq: -0.3467, KL: N/A, KL_t: 3.5021
[Epoch 26] Loss: 17.7213, FID: 45.0191, BCE: N/A
           MSE: 17.8287, Dq: -0.3561, KL: N/A, KL_t: 3.5289
[Epoch 27] Loss: 17.6658, FID: 44.8886, BCE: N/A
           MSE: 17.7767, Dq: -0.3692, KL: N/A, KL_t: 3.6853
[Epoch 28] Loss: 17.5751, FID: 43.7326, BCE: N/A
           MSE: 17.6825, Dq: -0.3580, KL: N/A, KL_t: 3.5827
[Epoch 29] Loss: 17.5315, FID: 44.8517, BCE: N/A
           MSE: 17.6351, Dq: -0.3564, KL: N/A, KL_t: 3.7302
[Epoch 30] Loss: 17.4623, FID: 44.6791, BCE: N/A
           MSE: 17.5699, Dq: -0.3587, KL: N/A, KL_t: 3.5875
[Epoch 31] Loss: 17.4268, FID: 44.0454, BCE: N/A
           MSE: 17.5312, Dq: -0.3621, KL: N/A, KL_t: 3.8311
[Epoch 32] Loss: 17

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-v7q0eriy'


[Epoch 62] Loss: 16.2764, FID: 37.2866, BCE: N/A
           MSE: 16.3706, Dq: -0.3479, KL: N/A, KL_t: 3.9916
[Epoch 63] Loss: 16.2267, FID: 37.8178, BCE: N/A
           MSE: 16.3259, Dq: -0.3465, KL: N/A, KL_t: 3.7016
[Epoch 64] Loss: 16.2315, FID: 38.1313, BCE: N/A
           MSE: 16.3270, Dq: -0.3475, KL: N/A, KL_t: 3.9161
[Epoch 65] Loss: 16.2190, FID: 37.7149, BCE: N/A
           MSE: 16.3162, Dq: -0.3498, KL: N/A, KL_t: 3.8839
[Epoch 66] Loss: 16.1890, FID: 37.7575, BCE: N/A
           MSE: 16.2787, Dq: -0.3393, KL: N/A, KL_t: 3.9993
[Epoch 67] Loss: 16.1672, FID: 37.4265, BCE: N/A
           MSE: 16.2583, Dq: -0.3385, KL: N/A, KL_t: 3.9086
[Epoch 68] Loss: 16.1282, FID: 37.6091, BCE: N/A
           MSE: 16.2119, Dq: -0.3208, KL: N/A, KL_t: 3.8327
[Epoch 69] Loss: 16.1278, FID: 37.2146, BCE: N/A
           MSE: 16.2084, Dq: -0.3237, KL: N/A, KL_t: 4.0616
[Epoch 70] Loss: 16.1052, FID: 37.6318, BCE: N/A
           MSE: 16.1901, Dq: -0.3334, KL: N/A, KL_t: 4.0861
[Epoch 71] Loss: 16

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-_i2n6qrc'


[Epoch 86] Loss: 15.8320, FID: 36.0059, BCE: N/A
           MSE: 15.9199, Dq: -0.3314, KL: N/A, KL_t: 3.8919
[Epoch 87] Loss: 15.8294, FID: 35.6006, BCE: N/A
           MSE: 15.9183, Dq: -0.3394, KL: N/A, KL_t: 4.0443
[Epoch 88] Loss: 15.8223, FID: 36.2015, BCE: N/A
           MSE: 15.9133, Dq: -0.3453, KL: N/A, KL_t: 4.0817
[Epoch 89] Loss: 15.8025, FID: 36.2601, BCE: N/A
           MSE: 15.8908, Dq: -0.3410, KL: N/A, KL_t: 4.1101
[Epoch 90] Loss: 15.7878, FID: 35.5271, BCE: N/A
           MSE: 15.8700, Dq: -0.3272, KL: N/A, KL_t: 4.0673
[Epoch 91] Loss: 15.7682, FID: 35.6247, BCE: N/A
           MSE: 15.8523, Dq: -0.3308, KL: N/A, KL_t: 4.0655
[Epoch 92] Loss: 15.7569, FID: 36.2799, BCE: N/A
           MSE: 15.8447, Dq: -0.3329, KL: N/A, KL_t: 3.9300
[Epoch 93] Loss: 15.7611, FID: 35.5847, BCE: N/A
           MSE: 15.8499, Dq: -0.3472, KL: N/A, KL_t: 4.2439
[Epoch 94] Loss: 15.7159, FID: 35.8189, BCE: N/A
           MSE: 15.8015, Dq: -0.3248, KL: N/A, KL_t: 3.8383
[Epoch 95] Loss: 15

### vae

In [18]:
#### ours
batch_size0 = 128
batch_size_test = 64
batch_size_test_nll = 16
epoch0 = 100
lr0 = 1e-4


sigma20 = 1
lambda_t0 = 0.1
lambda_KLt = 0.1

lambda_KL0 = 0
lambda_sum = 0.5 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1}
lambda_KL0 = 0 # vae的 KL，非vae时=0

torch.cuda.empty_cache()

device = torch.device(f"cuda:{cuda_num}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

flag_vae = True
flag_iwae = False
flag_cv = False
prefix = f"rerun5-F3-anew-latent{LATENT_DIM}"
folder = "/home/yzm/ipy/runs__re"
file_idx = 141
sample_K0 = 2

lambda_D0 = 0.5
lambda_sum = 1 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1} 的分母
num_KLt = 1 # 消融改这里

for lr0 in [1e-3]:
    for sample_K0 in [10]:  
        for qqq in range(5):    
            for tensround in range(file_idx, file_idx + 1):
                time_now = datetime.now().strftime("%m-%d_%H-%M")
                if flag_vae:
                    lambda_sum = 0
                    lambda_KLt = 1
                    lambda_KL0 = 1
                    model_dir=f"{folder}/{prefix}-{datasetname}-vae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir 
                    )
                elif flag_iwae:
                    model_dir=f"{folder}/{prefix}-{datasetname}-iwae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                elif flag_cv:
                    model_dir=f"{folder}/{prefix}-{datasetname}-cv-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                else:
                    model_dir=f"{folder}/{prefix}-{datasetname}-ours-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                    log_dir=model_dir
                )
                model = VAE().to(device)
                model_t = VAE().to(device)
                optimizer = torch.optim.Adam(model.parameters(), lr=lr0)

                fid_metric = FrechetInceptionDistance(feature=2048).to(device)

                data_loader = DataLoader(
                    train_dataset_torch, batch_size=batch_size0, shuffle=True,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                val_loader = DataLoader(
                    val_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4, worker_init_fn=seed_worker, generator=gen
                )
                                

                test_loader = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                test_loader_nll = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test_nll, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )

                plt_ck_p = epoch0 // 20  # 显示 20 次测试结果

                total_steps = epoch0 * len(data_loader)

                for epoch in range(epoch0):

                    losses = []
                    Dqs = []
                    KLs = []
                    KL_ts = []
                    MSEs = []
                    BCEs = []
                    sum_Dq_plus_lKLt_dot_KLt_s = []

                    alpha_max = 1.0
                    beta_max = 1.0
                    gamma_max = 1.0

                    warmup1 = 30      # 只用 MSE+KL
                    warmup2 = 80      # 完整过渡到 MSE+KL+BaM
                    # alpha = alpha_max * min(1.0, epoch / warmup1)
                    alpha = alpha_max
                    gamma = gamma_max
                    beta = beta_max
                    # beta = beta_max * min(
                    #     1.0, max(0.0, (epoch - warmup1) / (warmup2 - warmup1))
                    # )
                    # k 控制增长速度，k越小增长越慢

                    # k = 0.1
                    # epoch_since_warmup = max(0, epoch - warmup1)
                    # beta = beta_max * (1 - math.exp(-k * epoch_since_warmup))
                    # 确保在warmup2时beta接近beta_max

                    # 在 epoch 开头，保存 old encoder 参数
                    old_enc_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.encoder.state_dict().items()
                    }

                    for batch_idx, (x, y) in enumerate(data_loader):
                        x = x.to(device)  # [batch, ...]
                        recon_batch, mu, logvar = model(x)  # [batch, D]

                        # 恢复旧 encoder 的参数到临时 old_encoder 推断（在 CPU 拷贝上不需要梯度）
                        with torch.no_grad():
                            # 将保存的 old state 恢复到 model.encoder 的一个临时副本计算 mu_old/logvar_old
                            # 更高效：直接在当前 encoder 上 load_state_dict(old_enc_state)，但要小心线程安全与 GPU；这里用简单方式：
                            # 创建新 encoder 实例会更健壮但也更重 —— 下面采用直接把旧参数载入 model.encoder 并再恢复当前参数的办法（较快）
                            model_t.encoder.load_state_dict(old_enc_state)   # load old weights
                            # 注意：mu_old, logvar_old 都是在 no_grad 环境下计算的（无需保持 grad）
                            mu_old, logvar_old = model_t.encoder(x)         # compute old outputs

                        global_step = epoch * len(data_loader) + batch_idx

                        KL = 1.0 / x.size(0) * loss_function(x, recon_batch, mu, logvar)
                        KLs.append(KL.item())
                        
                        if not flag_vae and not flag_iwae:                            
                            if flag_cv:
                                Dq = cv_bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K1=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                            else:
                                Dq = bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                        else:
                            Dq = torch.tensor(0.0)
                        Dqs.append(Dq.item())
                        
                        MSE = nn.functional.mse_loss(
                            recon_batch, x, reduction="sum"
                        ) / x.size(0)

                        MSEs.append(MSE.item())
                        MSE = sigma20 * MSE / sigma20

                        var = torch.exp(logvar)
                        var_old = torch.exp(logvar_old)
                        # 先在 latent 维度 D 上求和
                        KL_t_per_latent = 0.5 * (
                            torch.log(var / var_old) +
                            (var_old + (mu_old - mu).pow(2)) / var - 1
                        ).sum(dim=1)                        # [N]
                        # 再对 batch 维度求均值
                        KL_t = KL_t_per_latent.mean()        # scalar

                        KL_ts.append(KL_t.item())

                        Dq *= lambda_D0
                        KL *= lambda_KL0
                        KL_t *= 2.0 / lambda_KLt

                        sum_Dq_plus_lKLt_dot_KLt = Dq + KL_t
                        sum_Dq_plus_lKLt_dot_KLt_s.append(sum_Dq_plus_lKLt_dot_KLt.item())
                        sum_Dq_plus_lKLt_dot_KLt *= lambda_sum
                        
                        if flag_iwae:
                            loss = 1 / x.size(0) * loss_iwae_gauss(mu, logvar, x, model.decode, k=sample_K0)
                        else:
                            loss = MSE + alpha * KL + beta * sum_Dq_plus_lKLt_dot_KLt

                        BCE = nn.functional.binary_cross_entropy(
                            recon_batch, x, reduction="sum"
                        )
                        BCE = BCE / x.size(0)
                        BCEs.append(BCE.item())

                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                        losses.append(loss.item())

                    # 3. 记录每个 epoch 的平均 loss
                    mean_loss = np.mean(losses)
                    mean_MSE = np.mean(MSEs)
                    mean_BCE = np.mean(BCEs)
                    mean_Dq = np.mean(Dqs)
                    mean_KL = np.mean(KLs)
                    mean_KL_t = np.mean(KL_ts)
                    mean_Dq_plus_lKLt_dot_KLt = np.mean(sum_Dq_plus_lKLt_dot_KLt_s)

                    writer.add_scalar("Loss/train_epoch", mean_loss, epoch)
                    writer.add_scalar("MSE/train_epoch", mean_MSE, epoch)
                    writer.add_scalar("BCE/train_epoch", mean_BCE, epoch)
                    writer.add_scalar("Dq/train_epoch", mean_Dq, epoch)
                    writer.add_scalar("KL/train_epoch", mean_KL, epoch)
                    writer.add_scalar("KL_t/train_epoch", mean_KL_t, epoch)
                    writer.add_scalar("Sum/train_epoch", mean_Dq_plus_lKLt_dot_KLt, epoch)

                    write_test_images(writer, model, test_loader, device, epoch)
                    
                    fid_score = eval_fid(model, val_loader, device, fid_metric)
                    writer.add_scalar("FID/val_epoch", fid_score, epoch)
                    print(
                        f"[Epoch {epoch+1}] Loss: {mean_loss:.4f}, FID: {fid_score:.4f}, BCE: {mean_BCE:.4f}"
                    )
                    print(
                        f"           MSE: {mean_MSE:.4f}, Dq: {mean_Dq:.4f}, KL: {mean_KL:.4f}, KL_t: {mean_KL_t:.4f}"
                    )
                    torch.cuda.empty_cache()
                    
                write_test_images(
                    writer,
                    model,
                    test_loader,
                    device,
                    global_step=0,
                    sample_K=42,
                    folder_name="img_final_epoch",
                )
                
                fid_score_final = eval_fid(model, test_loader, device, fid_metric)
                writer.add_scalar("FID/test_epoch", fid_score_final, epoch)
                print(f"Final Test FID: {fid_score_final:.4f}")

                hparam_dict = {
                    "batch_size": batch_size0,
                    "epochs": epoch0,
                    "lr": lr0,
                    "K": sample_K0,
                    "sigma2": sigma20,
                    "lambda_D": lambda_D0,
                    "lambda_KL": lambda_KL0,
                    "lambda_KLt": lambda_KLt,
                    "lambda_sum": lambda_sum,
                    "tensround": tensround,
                }
                metric_dict = {
                    "last loss": mean_loss,
                    "last FID": float(fid_score_final),
                    "last MSE": mean_MSE,
                }
                writer.add_hparams(hparam_dict, metric_dict)
                # 4. 训练结束后关闭 writer
                writer.close()
                
                torch.save(
                    model,
                    f"{model_dir}/final_model.pt",
                )
                
                # Lk, NLL, bpd = evaluate_iwae_gauss(model, test_loader_nll, device)
                # writer.add_scalar("NLL/test_final_epoch", NLL, 99)
                # writer.add_scalar("bpd/test_final_epoch", bpd, 99)


Using device: cuda:1


/home/yzm/anaconda3/envs/gen/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Metric `FrechetInceptionDistance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


[Epoch 1] Loss: 45.2489, FID: 162.2360, BCE: 173.7826
           MSE: 39.1152, Dq: 0.0000, KL: 6.1336, KL_t: 46.0003
[Epoch 2] Loss: 38.1950, FID: 127.2188, BCE: 148.0451
           MSE: 31.6348, Dq: 0.0000, KL: 6.5602, KL_t: 1.0908
[Epoch 3] Loss: 35.9686, FID: 114.1886, BCE: 139.8043
           MSE: 29.1429, Dq: 0.0000, KL: 6.8257, KL_t: 0.8766
[Epoch 4] Loss: 34.7528, FID: 102.8455, BCE: 134.9132
           MSE: 27.6783, Dq: 0.0000, KL: 7.0745, KL_t: 0.7428
[Epoch 5] Loss: 33.9719, FID: 94.5385, BCE: 131.6408
           MSE: 26.6776, Dq: 0.0000, KL: 7.2943, KL_t: 0.5858
[Epoch 6] Loss: 33.4175, FID: 91.0510, BCE: 129.3532
           MSE: 25.9610, Dq: 0.0000, KL: 7.4565, KL_t: 0.6366
[Epoch 7] Loss: 33.0021, FID: 84.2279, BCE: 127.6065
           MSE: 25.4174, Dq: 0.0000, KL: 7.5848, KL_t: 0.6654
[Epoch 8] Loss: 32.6960, FID: 82.9831, BCE: 126.3077
           MSE: 25.0059, Dq: 0.0000, KL: 7.6901, KL_t: 0.6026
[Epoch 9] Loss: 32.3911, FID: 79.4720, BCE: 125.1165
           MSE: 24.626

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-13_avaml'


[Epoch 45] Loss: 29.7726, FID: 53.3484, BCE: 114.3814
           MSE: 21.1193, Dq: 0.0000, KL: 8.6533, KL_t: 0.6368
[Epoch 46] Loss: 29.7666, FID: 54.3519, BCE: 114.3045
           MSE: 21.0987, Dq: 0.0000, KL: 8.6679, KL_t: 0.7788
[Epoch 47] Loss: 29.7627, FID: 52.6821, BCE: 114.2197
           MSE: 21.0666, Dq: 0.0000, KL: 8.6961, KL_t: 0.6733
[Epoch 48] Loss: 29.7033, FID: 54.3594, BCE: 114.0293
           MSE: 21.0069, Dq: 0.0000, KL: 8.6965, KL_t: 0.7937
[Epoch 49] Loss: 29.6911, FID: 52.9955, BCE: 114.0069
           MSE: 20.9895, Dq: 0.0000, KL: 8.7017, KL_t: 0.7622
[Epoch 50] Loss: 29.6888, FID: 52.1804, BCE: 113.8979
           MSE: 20.9719, Dq: 0.0000, KL: 8.7169, KL_t: 0.7326
[Epoch 51] Loss: 29.6582, FID: 51.0127, BCE: 113.7709
           MSE: 20.9242, Dq: 0.0000, KL: 8.7340, KL_t: 0.6130
[Epoch 52] Loss: 29.6486, FID: 53.3590, BCE: 113.7515
           MSE: 20.9170, Dq: 0.0000, KL: 8.7316, KL_t: 0.6908
[Epoch 53] Loss: 29.6033, FID: 50.7777, BCE: 113.6373
           MSE: 20

...

[Epoch 25] Loss: 30.6363, FID: 59.7304, BCE: 117.6769
           MSE: 22.2391, Dq: 0.0000, KL: 8.3972, KL_t: 0.6458
[Epoch 26] Loss: 30.5547, FID: 59.3487, BCE: 117.3716
           MSE: 22.1334, Dq: 0.0000, KL: 8.4213, KL_t: 0.6390
[Epoch 27] Loss: 30.4865, FID: 58.2059, BCE: 117.1767
           MSE: 22.0683, Dq: 0.0000, KL: 8.4182, KL_t: 0.7693
[Epoch 28] Loss: 30.4673, FID: 56.9412, BCE: 117.0140
           MSE: 22.0115, Dq: 0.0000, KL: 8.4558, KL_t: 0.6827
[Epoch 29] Loss: 30.4065, FID: 58.2222, BCE: 116.7732
           MSE: 21.9334, Dq: 0.0000, KL: 8.4732, KL_t: 0.6772
[Epoch 30] Loss: 30.3513, FID: 56.8461, BCE: 116.5761
           MSE: 21.8669, Dq: 0.0000, KL: 8.4844, KL_t: 0.7465
[Epoch 31] Loss: 30.3464, FID: 58.5198, BCE: 116.4326
           MSE: 21.8299, Dq: 0.0000, KL: 8.5165, KL_t: 0.8199
[Epoch 32] Loss: 30.2704, FID: 56.6281, BCE: 116.2117
           MSE: 21.7535, Dq: 0.0000, KL: 8.5169, KL_t: 0.7018
[Epoch 33] Loss: 30.2591, FID: 56.5437, BCE: 116.0938
           MSE: 21

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-66t9trfz'


[Epoch 57] Loss: 29.5927, FID: 52.0839, BCE: 113.3061
           MSE: 20.7816, Dq: 0.0000, KL: 8.8112, KL_t: 0.7087
[Epoch 58] Loss: 29.5824, FID: 51.1077, BCE: 113.2770
           MSE: 20.7662, Dq: 0.0000, KL: 8.8162, KL_t: 0.6754
[Epoch 59] Loss: 29.5706, FID: 50.7234, BCE: 113.1905
           MSE: 20.7429, Dq: 0.0000, KL: 8.8277, KL_t: 0.7364
[Epoch 60] Loss: 29.5179, FID: 50.9973, BCE: 113.0676
           MSE: 20.7038, Dq: 0.0000, KL: 8.8141, KL_t: 0.7461
[Epoch 61] Loss: 29.5126, FID: 50.9500, BCE: 113.0665
           MSE: 20.6938, Dq: 0.0000, KL: 8.8189, KL_t: 0.7753
[Epoch 62] Loss: 29.5074, FID: 50.8453, BCE: 112.9487
           MSE: 20.6594, Dq: 0.0000, KL: 8.8480, KL_t: 0.8655
[Epoch 63] Loss: 29.4622, FID: 52.1657, BCE: 112.8871
           MSE: 20.6325, Dq: 0.0000, KL: 8.8297, KL_t: 0.6034
[Epoch 64] Loss: 29.4475, FID: 51.5168, BCE: 112.7841
           MSE: 20.5981, Dq: 0.0000, KL: 8.8494, KL_t: 0.7927
[Epoch 65] Loss: 29.4640, FID: 51.2249, BCE: 112.8033
           MSE: 20

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-pvluge5v'


[Epoch 87] Loss: 29.1669, FID: 48.9024, BCE: 111.6127
           MSE: 20.1948, Dq: 0.0000, KL: 8.9720, KL_t: 0.7590
[Epoch 88] Loss: 29.1777, FID: 49.6422, BCE: 111.6245
           MSE: 20.2035, Dq: 0.0000, KL: 8.9743, KL_t: 0.7852
[Epoch 89] Loss: 29.1501, FID: 50.2427, BCE: 111.4966
           MSE: 20.1600, Dq: 0.0000, KL: 8.9901, KL_t: 0.8273
[Epoch 90] Loss: 29.1525, FID: 48.7133, BCE: 111.5122
           MSE: 20.1632, Dq: 0.0000, KL: 8.9893, KL_t: 0.7437
[Epoch 91] Loss: 29.1385, FID: 49.1607, BCE: 111.4729
           MSE: 20.1459, Dq: 0.0000, KL: 8.9925, KL_t: 0.6923
[Epoch 92] Loss: 29.1191, FID: 49.3686, BCE: 111.3887
           MSE: 20.1179, Dq: 0.0000, KL: 9.0012, KL_t: 0.7142
[Epoch 93] Loss: 29.1362, FID: 48.5001, BCE: 111.4261
           MSE: 20.1315, Dq: 0.0000, KL: 9.0047, KL_t: 0.6902
[Epoch 94] Loss: 29.1189, FID: 47.5679, BCE: 111.3574
           MSE: 20.1084, Dq: 0.0000, KL: 9.0105, KL_t: 0.7313
[Epoch 95] Loss: 29.0884, FID: 48.6435, BCE: 111.2918
           MSE: 20

### iwae

In [ ]:
#### ours
batch_size0 = 128
batch_size_test = 64
batch_size_test_nll = 16
epoch0 = 100
lr0 = 1e-4


sigma20 = 1
lambda_t0 = 0.1
lambda_KLt = 0.1

lambda_KL0 = 0
lambda_sum = 0.5 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1}
lambda_KL0 = 0 # vae的 KL，非vae时=0

torch.cuda.empty_cache()

device = torch.device(f"cuda:{cuda_num}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

flag_vae = False
flag_iwae = True
flag_cv = False
prefix = f"rerun5-F3-anew-latent{LATENT_DIM}"
folder = "/home/yzm/ipy/runs__re"
file_idx = 141
sample_K0 = 2

lambda_D0 = 0.5
lambda_sum = 1 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1} 的分母
num_KLt = 1 # 消融改这里

for lr0 in [1e-3]:
    for sample_K0 in [10]:  
        for qqq in range(5):    
            for tensround in range(file_idx, file_idx + 1):
                time_now = datetime.now().strftime("%m-%d_%H-%M")
                if flag_vae:
                    lambda_sum = 0
                    lambda_KLt = 1
                    lambda_KL0 = 1
                    model_dir=f"{folder}/{prefix}-{datasetname}-vae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir 
                    )
                elif flag_iwae:
                    model_dir=f"{folder}/{prefix}-{datasetname}-iwae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                elif flag_cv:
                    model_dir=f"{folder}/{prefix}-{datasetname}-cv-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                else:
                    model_dir=f"{folder}/{prefix}-{datasetname}-ours-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                    log_dir=model_dir
                )
                model = VAE().to(device)
                model_t = VAE().to(device)
                optimizer = torch.optim.Adam(model.parameters(), lr=lr0)

                fid_metric = FrechetInceptionDistance(feature=2048).to(device)

                data_loader = DataLoader(
                    train_dataset_torch, batch_size=batch_size0, shuffle=True,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                val_loader = DataLoader(
                    val_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4, worker_init_fn=seed_worker, generator=gen
                )
                                

                test_loader = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                test_loader_nll = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test_nll, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )

                plt_ck_p = epoch0 // 20  # 显示 20 次测试结果

                total_steps = epoch0 * len(data_loader)

                for epoch in range(epoch0):

                    losses = []
                    Dqs = []
                    KLs = []
                    KL_ts = []
                    MSEs = []
                    BCEs = []
                    sum_Dq_plus_lKLt_dot_KLt_s = []

                    alpha_max = 1.0
                    beta_max = 1.0
                    gamma_max = 1.0

                    warmup1 = 30      # 只用 MSE+KL
                    warmup2 = 80      # 完整过渡到 MSE+KL+BaM
                    # alpha = alpha_max * min(1.0, epoch / warmup1)
                    alpha = alpha_max
                    gamma = gamma_max
                    beta = beta_max
                    # beta = beta_max * min(
                    #     1.0, max(0.0, (epoch - warmup1) / (warmup2 - warmup1))
                    # )
                    # k 控制增长速度，k越小增长越慢

                    # k = 0.1
                    # epoch_since_warmup = max(0, epoch - warmup1)
                    # beta = beta_max * (1 - math.exp(-k * epoch_since_warmup))
                    # 确保在warmup2时beta接近beta_max

                    # 在 epoch 开头，保存 old encoder 参数
                    old_enc_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.encoder.state_dict().items()
                    }

                    for batch_idx, (x, y) in enumerate(data_loader):
                        x = x.to(device)  # [batch, ...]
                        recon_batch, mu, logvar = model(x)  # [batch, D]

                        # 恢复旧 encoder 的参数到临时 old_encoder 推断（在 CPU 拷贝上不需要梯度）
                        with torch.no_grad():
                            # 将保存的 old state 恢复到 model.encoder 的一个临时副本计算 mu_old/logvar_old
                            # 更高效：直接在当前 encoder 上 load_state_dict(old_enc_state)，但要小心线程安全与 GPU；这里用简单方式：
                            # 创建新 encoder 实例会更健壮但也更重 —— 下面采用直接把旧参数载入 model.encoder 并再恢复当前参数的办法（较快）
                            model_t.encoder.load_state_dict(old_enc_state)   # load old weights
                            # 注意：mu_old, logvar_old 都是在 no_grad 环境下计算的（无需保持 grad）
                            mu_old, logvar_old = model_t.encoder(x)         # compute old outputs

                        global_step = epoch * len(data_loader) + batch_idx

                        KL = 1.0 / x.size(0) * loss_function(x, recon_batch, mu, logvar)
                        KLs.append(KL.item())
                        
                        if not flag_vae and not flag_iwae:                            
                            if flag_cv:
                                Dq = cv_bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K1=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                            else:
                                Dq = bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                        else:
                            Dq = torch.tensor(0.0)
                        Dqs.append(Dq.item())
                        
                        MSE = nn.functional.mse_loss(
                            recon_batch, x, reduction="sum"
                        ) / x.size(0)

                        MSEs.append(MSE.item())
                        MSE = sigma20 * MSE / sigma20

                        var = torch.exp(logvar)
                        var_old = torch.exp(logvar_old)
                        # 先在 latent 维度 D 上求和
                        KL_t_per_latent = 0.5 * (
                            torch.log(var / var_old) +
                            (var_old + (mu_old - mu).pow(2)) / var - 1
                        ).sum(dim=1)                        # [N]
                        # 再对 batch 维度求均值
                        KL_t = KL_t_per_latent.mean()        # scalar

                        KL_ts.append(KL_t.item())

                        Dq *= lambda_D0
                        KL *= lambda_KL0
                        KL_t *= 2.0 / lambda_KLt

                        sum_Dq_plus_lKLt_dot_KLt = Dq + KL_t
                        sum_Dq_plus_lKLt_dot_KLt_s.append(sum_Dq_plus_lKLt_dot_KLt.item())
                        sum_Dq_plus_lKLt_dot_KLt *= lambda_sum
                        
                        if flag_iwae:
                            loss = 1 / x.size(0) * loss_iwae_gauss(mu, logvar, x, model.decode, k=sample_K0)
                        else:
                            loss = MSE + alpha * KL + beta * sum_Dq_plus_lKLt_dot_KLt

                        BCE = nn.functional.binary_cross_entropy(
                            recon_batch, x, reduction="sum"
                        )
                        BCE = BCE / x.size(0)
                        BCEs.append(BCE.item())

                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                        losses.append(loss.item())

                    # 3. 记录每个 epoch 的平均 loss
                    mean_loss = np.mean(losses)
                    mean_MSE = np.mean(MSEs)
                    mean_BCE = np.mean(BCEs)
                    mean_Dq = np.mean(Dqs)
                    mean_KL = np.mean(KLs)
                    mean_KL_t = np.mean(KL_ts)
                    mean_Dq_plus_lKLt_dot_KLt = np.mean(sum_Dq_plus_lKLt_dot_KLt_s)

                    writer.add_scalar("Loss/train_epoch", mean_loss, epoch)
                    writer.add_scalar("MSE/train_epoch", mean_MSE, epoch)
                    writer.add_scalar("BCE/train_epoch", mean_BCE, epoch)
                    writer.add_scalar("Dq/train_epoch", mean_Dq, epoch)
                    writer.add_scalar("KL/train_epoch", mean_KL, epoch)
                    writer.add_scalar("KL_t/train_epoch", mean_KL_t, epoch)
                    writer.add_scalar("Sum/train_epoch", mean_Dq_plus_lKLt_dot_KLt, epoch)

                    write_test_images(writer, model, test_loader, device, epoch)
                    
                    fid_score = eval_fid(model, val_loader, device, fid_metric)
                    writer.add_scalar("FID/val_epoch", fid_score, epoch)
                    print(
                        f"[Epoch {epoch+1}] Loss: {mean_loss:.4f}, FID: {fid_score:.4f}, BCE: {mean_BCE:.4f}"
                    )
                    print(
                        f"           MSE: {mean_MSE:.4f}, Dq: {mean_Dq:.4f}, KL: {mean_KL:.4f}, KL_t: {mean_KL_t:.4f}"
                    )
                    torch.cuda.empty_cache()
                    
                write_test_images(
                    writer,
                    model,
                    test_loader,
                    device,
                    global_step=0,
                    sample_K=42,
                    folder_name="img_final_epoch",
                )
                
                fid_score_final = eval_fid(model, test_loader, device, fid_metric)
                writer.add_scalar("FID/test_epoch", fid_score_final, epoch)
                print(f"Final Test FID: {fid_score_final:.4f}")

                hparam_dict = {
                    "batch_size": batch_size0,
                    "epochs": epoch0,
                    "lr": lr0,
                    "K": sample_K0,
                    "sigma2": sigma20,
                    "lambda_D": lambda_D0,
                    "lambda_KL": lambda_KL0,
                    "lambda_KLt": lambda_KLt,
                    "lambda_sum": lambda_sum,
                    "tensround": tensround,
                }
                metric_dict = {
                    "last loss": mean_loss,
                    "last FID": float(fid_score_final),
                    "last MSE": mean_MSE,
                }
                writer.add_hparams(hparam_dict, metric_dict)
                # 4. 训练结束后关闭 writer
                writer.close()
                
                torch.save(
                    model,
                    f"{model_dir}/final_model.pt",
                )
                
                # Lk, NLL, bpd = evaluate_iwae_gauss(model, test_loader_nll, device)
                # writer.add_scalar("NLL/test_final_epoch", NLL, 99)
                # writer.add_scalar("bpd/test_final_epoch", bpd, 99)


Using device: cuda:1


[Epoch 1] Loss: 744.8348, FID: 166.1795, BCE: 195.9366
           MSE: 46.4628, Dq: 0.0000, KL: 3.0309, KL_t: 8.9811
[Epoch 2] Loss: 741.5966, FID: 143.1968, BCE: 165.4546
           MSE: 37.9667, Dq: 0.0000, KL: 3.3360, KL_t: 0.6973
[Epoch 3] Loss: 740.7309, FID: 122.1783, BCE: 160.3708
           MSE: 36.6324, Dq: 0.0000, KL: 3.5297, KL_t: 0.4113
[Epoch 4] Loss: 740.2935, FID: 114.3738, BCE: 157.2495
           MSE: 35.6878, Dq: 0.0000, KL: 3.7131, KL_t: 0.5849
[Epoch 5] Loss: 740.0078, FID: 107.7879, BCE: 155.2135
           MSE: 35.0109, Dq: 0.0000, KL: 3.8388, KL_t: 0.4846
[Epoch 6] Loss: 739.8084, FID: 101.7036, BCE: 153.2976
           MSE: 34.3585, Dq: 0.0000, KL: 3.9680, KL_t: 0.5400
[Epoch 7] Loss: 739.6609, FID: 94.5703, BCE: 152.2051
           MSE: 33.9888, Dq: 0.0000, KL: 4.0595, KL_t: 0.5092
[Epoch 8] Loss: 739.5404, FID: 92.9013, BCE: 150.8867
           MSE: 33.5698, Dq: 0.0000, KL: 4.1499, KL_t: 0.5265
[Epoch 9] Loss: 739.4362, FID: 89.4271, BCE: 149.8333
           M

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-7vx2h3y_'


[Epoch 70] Loss: 738.3457, FID: 61.0875, BCE: 141.1727
           MSE: 30.0779, Dq: 0.0000, KL: 5.0729, KL_t: 0.6016
[Epoch 71] Loss: 738.3385, FID: 60.5944, BCE: 141.4340
           MSE: 30.1714, Dq: 0.0000, KL: 5.0420, KL_t: 0.5623
[Epoch 72] Loss: 738.3267, FID: 57.6537, BCE: 141.5152
           MSE: 30.1957, Dq: 0.0000, KL: 5.0433, KL_t: 0.6277
[Epoch 73] Loss: 738.3386, FID: 59.8809, BCE: 140.8138
           MSE: 29.9612, Dq: 0.0000, KL: 5.0862, KL_t: 0.6411
[Epoch 74] Loss: 738.3325, FID: 58.3758, BCE: 141.2176
           MSE: 30.0890, Dq: 0.0000, KL: 5.0779, KL_t: 0.5738
[Epoch 75] Loss: 738.3211, FID: 59.2786, BCE: 141.2222
           MSE: 30.0480, Dq: 0.0000, KL: 5.0502, KL_t: 0.5810
[Epoch 76] Loss: 738.3230, FID: 59.6565, BCE: 140.7774
           MSE: 29.9375, Dq: 0.0000, KL: 5.1023, KL_t: 0.5013


Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-wlt1uxa5'


[Epoch 77] Loss: 738.3134, FID: 57.5973, BCE: 140.7936
           MSE: 29.9192, Dq: 0.0000, KL: 5.1010, KL_t: 0.5553
[Epoch 78] Loss: 738.3271, FID: 57.9385, BCE: 141.1954
           MSE: 30.0415, Dq: 0.0000, KL: 5.1041, KL_t: 0.6207
[Epoch 79] Loss: 738.3130, FID: 57.7834, BCE: 141.3331
           MSE: 30.0891, Dq: 0.0000, KL: 5.0737, KL_t: 0.5995
[Epoch 80] Loss: 738.3034, FID: 59.8311, BCE: 140.6868
           MSE: 29.8802, Dq: 0.0000, KL: 5.1248, KL_t: 0.5849
[Epoch 81] Loss: 738.3085, FID: 57.4826, BCE: 141.1564
           MSE: 30.0320, Dq: 0.0000, KL: 5.0884, KL_t: 0.6152
[Epoch 82] Loss: 738.2884, FID: 59.7756, BCE: 140.6415
           MSE: 29.8939, Dq: 0.0000, KL: 5.1125, KL_t: 0.5406
[Epoch 83] Loss: 738.3011, FID: 57.6264, BCE: 140.5694
           MSE: 29.8713, Dq: 0.0000, KL: 5.1129, KL_t: 0.5847
[Epoch 84] Loss: 738.2966, FID: 58.2330, BCE: 140.9043
           MSE: 29.9549, Dq: 0.0000, KL: 5.1132, KL_t: 0.6992
[Epoch 85] Loss: 738.2935, FID: 60.1626, BCE: 140.3442
         

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-8y8uqfs8'


[Epoch 89] Loss: 738.2779, FID: 58.8509, BCE: 140.7624
           MSE: 29.9098, Dq: 0.0000, KL: 5.1213, KL_t: 0.5472
[Epoch 90] Loss: 738.2810, FID: 60.1392, BCE: 140.9507
           MSE: 29.9651, Dq: 0.0000, KL: 5.1179, KL_t: 0.5207
[Epoch 91] Loss: 738.2689, FID: 56.5205, BCE: 140.7250
           MSE: 29.8710, Dq: 0.0000, KL: 5.1294, KL_t: 0.5783
[Epoch 92] Loss: 738.2811, FID: 56.3583, BCE: 140.4161
           MSE: 29.7993, Dq: 0.0000, KL: 5.1543, KL_t: 0.6405
[Epoch 93] Loss: 738.2639, FID: 58.5951, BCE: 140.5504
           MSE: 29.8281, Dq: 0.0000, KL: 5.1309, KL_t: 0.5319
[Epoch 94] Loss: 738.2631, FID: 58.7941, BCE: 140.2328
           MSE: 29.7411, Dq: 0.0000, KL: 5.1645, KL_t: 0.5828
[Epoch 95] Loss: 738.2690, FID: 56.0780, BCE: 140.0723
           MSE: 29.6777, Dq: 0.0000, KL: 5.1608, KL_t: 0.5936
[Epoch 96] Loss: 738.2666, FID: 58.6058, BCE: 140.6379
           MSE: 29.8352, Dq: 0.0000, KL: 5.1357, KL_t: 0.5497
[Epoch 97] Loss: 738.2497, FID: 57.2209, BCE: 140.0463
         

## eof